# Day 39: KNN Imputer | Multivariate Imputation | Handling Missing Data Part 5

src: https://www.youtube.com/watch?v=-fK-xEev2I8&

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.impute import KNNImputer,SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score

In [2]:
df = pd.read_csv('train.csv')[['Age','Pclass','Fare','Survived']]

In [12]:
df.tail()

,Age,Pclass,Fare,Survived
886,27.0,2,13.00,0
887,19.0,1,30.00,1
888,NaN,3,23.45,0
889,26.0,1,30.00,1
890,32.0,3,7.75,0


In [4]:
df.isnull().mean() * 100

,0
Age,19.86532
Pclass,0.00000
Fare,0.00000
Survived,0.00000


---

| Feature      | % Missing  | Explanation                                                                                                                                                                       |
| ------------ | ---------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Age**      | **19.87%** | Roughly **20% of rows are missing Age values**. This is common in Titanic datasets where age was not recorded for all passengers. Imputation is needed (e.g., mean, median, KNN). |
| **Pclass**   | **0.00%**  | No missing values in the **Pclass** (passenger class) column. Fully available for use.                                                                                            |
| **Fare**     | **0.00%**  | The **Fare** column (ticket price) has no missing values.                                                                                                                         |
| **Survived** | **0.00%**  | No missing values in the target column **Survived**. Ready for modeling.                                                                                                          |

---



In [5]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [6]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=2)

In [13]:
X_train.tail()

,Age,Pclass,Fare
534,30.0,3,8.6625
584,NaN,3,8.7125
493,71.0,1,49.5042
527,NaN,1,221.7792
168,NaN,1,25.9250


## **Comparing KNN Imputer vs SimpleImputer (Mean values)**

### Using KNN Imputer

In [8]:
from sklearn.impute import KNNImputer

# Step 1: Create a KNN Imputer object
# n_neighbors=3 --> Use 3 nearest rows (neighbors) to fill in missing values
# weights='distance' --> Closer neighbors are given more influence (higher weight) than distant ones
knn = KNNImputer(n_neighbors=3, weights='distance')

# Step 2: Fit the imputer on the training data and transform it
# This step computes the distances between rows in X_train
# Then fills missing values using the values of the 3 closest rows
X_train_trf = knn.fit_transform(X_train)

# Step 3: Use the same imputer (already fitted on training data) to transform the test set
# This ensures consistency: missing values in X_test are filled based on training patterns
X_test_trf = knn.transform(X_test)

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Step 1: Create a Logistic Regression model
lr = LogisticRegression()

# Step 2: Train (fit) the model on the imputed training data
# X_train_trf → Feature matrix with no missing values (after KNN imputation)
# y_train → Actual labels (0 or 1 for classification)
lr.fit(X_train_trf, y_train)

# Step 3: Use the trained model to make predictions on the test data
# X_test_trf → Test features (also imputed using the same KNNImputer)
y_pred = lr.predict(X_test_trf)

# Step 4: Evaluate the model’s performance using accuracy
# Compares predicted labels with actual test labels
accuracy_score(y_test, y_pred)

0.7039106145251397

---

### ✅ Interpretation:

* **Accuracy ≈ 70.39%**
* This means the model correctly predicted survival for \~70% of the passengers in the test set.
* It’s a reasonable baseline, especially for Titanic-like datasets.

---

🔎 **Experiment Further:**

* Try other models (e.g., DecisionTree, RandomForest)?
* Add a confusion matrix or precision/recall scores?
* Run hyperparameter tuning for better performance?

---


### **Using SimpleImputer (Impute with Mean values)**

In [10]:
from sklearn.impute import SimpleImputer

# Step 1: Create a Simple Imputer
# By default, strategy='mean' → fill missing values with the column mean
si = SimpleImputer()

# Step 2: Fit the imputer on the training data and transform it
# Fills missing values in each column of X_train with that column's mean
X_train_trf2 = si.fit_transform(X_train)

# Step 3: Transform the test data using the same column means from training
# Ensures that test data is treated consistently
X_test_trf2 = si.transform(X_test)

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Step 1: Create a new logistic regression model
lr = LogisticRegression()

# Step 2: Train it on the mean-imputed training data
lr.fit(X_train_trf2, y_train)

# Step 3: Predict outcomes on the mean-imputed test data
y_pred2 = lr.predict(X_test_trf2)

# Step 4: Evaluate accuracy
accuracy_score(y_test, y_pred2)


0.6927374301675978

---

### 📊 Logistic Regression Accuracy: KNN Imputer vs Simple (Mean) Imputer

| Imputation Method                           | Accuracy (%)          |
| ------------------------------------------- | --------------------- |
| **KNN Imputer** (k=3, weighted by distance) | **70.39%** (`0.7039`) |
| **Simple Imputer** (Mean)                   | **69.27%** (`0.6927`) |

---

## ✅ Interpretation:

* The **KNN Imputer** performed **slightly better** than the Simple (mean) imputer.
* This suggests that **using values from similar rows (neighbors)** to fill missing values helped the model learn patterns more effectively.
* The difference (\~1.1%) is small but can become more significant depending on:

  * Dataset size
  * Number of missing values
  * Nature of the missingness (MCAR, MAR, MNAR)

---

## 🧠 When to Prefer KNN Imputation:

| Use KNN If...                                      | Because...                        |
| -------------------------------------------------- | --------------------------------- |
| You have moderate dataset size                     | KNN is computationally heavier    |
| You want to preserve relationships between columns | KNN uses **multivariate context** |
| You care about model accuracy over training speed  | KNN is slower but smarter         |

---

🔎 **Experiment Further**

Would you like a visual bar chart or try more imputers (e.g., `IterativeImputer`, `Median`, or `Random`) for comparison?

---